In [1]:
# Creating the XGboost training cell
import pandas as pd
import numpy as np
import joblib
import time

from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

DATA_DIR = Path("../data/processed")
MODEL_DIR = Path("../models")
REPORT_DIR = Path("../reports")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Load the same prepared dataset
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")

y_train = pd.read_csv(DATA_DIR / "y_train.csv")[
    "disease_encoded"
]
y_test = pd.read_csv(DATA_DIR / "y_test.csv")[
    "disease_encoded"
]

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Number of training classes:", y_train.nunique())

Training shape: (80000, 174)
Testing shape: (20000, 174)
Number of training classes: 46


In [2]:
# Training the model
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=46,
    n_estimators=300,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1
)

start_time = time.time()

xgb_model.fit(X_train, y_train)

training_time = time.time() - start_time

print(f"Training completed in {training_time:.2f} seconds")

Training completed in 92.11 seconds


In [3]:
# Evaluating the model
y_pred_xgb = xgb_model.predict(X_test)

xgb_metrics = {
    "Model": "XGBoost",
    "Accuracy": accuracy_score(y_test, y_pred_xgb),
    "Macro F1": f1_score(
        y_test, y_pred_xgb,
        average="macro",
        zero_division=0
    ),
    "Weighted F1": f1_score(
        y_test, y_pred_xgb,
        average="weighted",
        zero_division=0
    ),
    "Training Time (seconds)": training_time
}

for metric, value in xgb_metrics.items():
    if isinstance(value, (float, np.floating)):
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value}")

Model: XGBoost
Accuracy: 0.6671
Macro F1: 0.3446
Weighted F1: 0.6577
Training Time (seconds): 92.1111


In [4]:
# Saving the model
joblib.dump(
    xgb_model,
    MODEL_DIR / "xgboost_model.joblib"
)

pd.DataFrame([xgb_metrics]).to_csv(
    REPORT_DIR / "xgboost_metrics.csv",
    index=False
)

print("XGBoost model and metrics saved successfully.")

XGBoost model and metrics saved successfully.


In [5]:
# Comparing the models Random Forest and XGboost
rf_metrics = pd.read_csv(
    REPORT_DIR / "random_forest_metrics.csv"
).iloc[0]

xgb_saved_metrics = pd.read_csv(
    REPORT_DIR / "xgboost_metrics.csv"
).iloc[0]

comparison = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Accuracy": rf_metrics["Accuracy"],
        "Macro F1": rf_metrics["Macro F1"],
        "Weighted F1": rf_metrics["Weighted F1"]
    },
    {
        "Model": "XGBoost",
        "Accuracy": xgb_saved_metrics["Accuracy"],
        "Macro F1": xgb_saved_metrics["Macro F1"],
        "Weighted F1": xgb_saved_metrics["Weighted F1"]
    }
])

display(comparison.round(4))

,Model,Accuracy,Macro F1,Weighted F1
0,Random Forest,0.3523,0.2229,0.3567
1,XGBoost,0.6671,0.3446,0.6577
